1. Load the dataset
2. Select useful features
3. Remove identifier columns
4. Encode categorical data
5. Separate features and target
6. Split the dataset into training and testing sets
7. Scale features for Logistic Regression
8. Save the preprocessed data and scaler

In [18]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [19]:
df= pd.read_csv("../data/ai4i2020.csv")
df.head(1)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0


In [20]:
# Select Useful Features

features = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

target = "Machine failure"

x=df[features].copy()
y=df[target].copy()
print("Features Shape:", x.shape)
print("Target Shape:", y.shape)

Features Shape: (10000, 6)
Target Shape: (10000,)


X contains the machine-condition features used for prediction.

y contains the target we want to predict.

UDI and Product ID are excluded because they are identifiers.

Failure-mode indicators (TWF, HDF, PWF, OSF, RNF) are excluded to avoid using information about specific failure events as model inputs.

In [21]:
x.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


In [22]:
y.value_counts()

Machine failure
0    9661
1     339
Name: count, dtype: int64

In [23]:
x = pd.get_dummies(
    x,columns=['Type'],
    drop_first=True,
    dtype=int

)
x.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,1
1,298.2,308.7,1408,46.3,3,1,0
2,298.1,308.5,1498,49.4,5,1,0
3,298.2,308.6,1433,39.5,7,1,0
4,298.2,308.7,1408,40.0,9,1,0


In [24]:
# Check Encoded Features

print("feature columns")
print(x.columns.tolist())

print("\n feature data type")
print(x.dtypes)

feature columns
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_L', 'Type_M']

 feature data type
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Type_L                       int64
Type_M                       int64
dtype: object


In [25]:
# Split Training and Testing Data

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.20,random_state=42,stratify=y)

print("X_train Shape:", x_train.shape)
print("X_test Shape:", x_test.shape)

print("y_train Shape:", y_train.shape)
print("y_test Shape:", y_test.shape)

X_train Shape: (8000, 7)
X_test Shape: (2000, 7)
y_train Shape: (8000,)
y_test Shape: (2000,)


In [26]:
# Check Target Distribution After Splitting

print("trainig target distribution")
print(y_train.value_counts())

print("testing target distribution")
print(y_test.value_counts())

trainig target distribution
Machine failure
0    7729
1     271
Name: count, dtype: int64
testing target distribution
Machine failure
0    1932
1      68
Name: count, dtype: int64


In [27]:
# Feature Scaling

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)
print("Scaled Training Shape:", x_train_scaled.shape)
print("Scaled Testing Shape:", x_test_scaled.shape)

Scaled Training Shape: (8000, 7)
Scaled Testing Shape: (2000, 7)


In [28]:
# Check Scaled Data

print("First 5 Scaled Training Records:")

print(x_train_scaled[:5])

First 5 Scaled Training Records:
[[ 0.99891359  0.60428162 -0.46060741  0.71830476 -0.84399726 -1.23436595
   1.53620728]
 [-1.50519408 -1.15325984 -0.77557391  0.63845583  0.38226308 -1.23436595
   1.53620728]
 [ 0.49809206  1.07746586 -1.00765448  0.5586069   0.46086951 -1.23436595
   1.53620728]
 [-0.55363316 -0.13929361 -0.70926517  1.62658635 -0.37235867  0.81013252
  -0.65095382]
 [-1.45511192 -1.01806434  1.07001923 -1.12820177 -0.90688241  0.81013252
  -0.65095382]]


In [29]:
os.makedirs("../models", exist_ok=True)

print("Models folder is ready.")

Models folder is ready.


In [30]:
joblib.dump(scaler, "../models/scaler.pkl")

print("Scaler saved successfully.")

Scaler saved successfully.


In [31]:
preprocessed_data = {
    "X_train": x_train,
    "X_test": x_test,
    "y_train": y_train,
    "y_test": y_test,
    "X_train_scaled": x_train_scaled,
    "X_test_scaled": x_test_scaled,
    "feature_columns": x.columns.tolist()
}

joblib.dump(
    preprocessed_data,
    "../models/preprocessed_data.pkl"
)

print("Preprocessed data saved successfully.")

Preprocessed data saved successfully.


In [32]:
print("Saved Files:")

print(os.listdir("../models"))

Saved Files:
['preprocessed_data.pkl', 'scaler.pkl']


In [33]:
print("========== PHASE 3 SUMMARY ==========")

print("Original Dataset Shape:", df.shape)
print("Selected Features:", x.shape[1])

print("Training Records:", len(x_train))
print("Testing Records:", len(x_test))

print("Training Failure Counts:")
print(y_train.value_counts())

print("\nTesting Failure Counts:")
print(y_test.value_counts())

print("\nScaling completed for Logistic Regression.")
print("Preprocessed data saved successfully.")

========== PHASE 3 SUMMARY ==========
Original Dataset Shape: (10000, 14)
Selected Features: 7
Training Records: 8000
Testing Records: 2000
Training Failure Counts:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Testing Failure Counts:
Machine failure
0    1932
1      68
Name: count, dtype: int64

Scaling completed for Logistic Regression.
Preprocessed data saved successfully.
